# LeetCode 3830 — Longest Alternating Subarray After Removing At Most One Element bằng Python

- Link: https://leetcode.com/problems/longest-alternating-subarray-after-removing-at-most-one-element/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên mảng, tách tiền tố–hậu tố (prefix–suffix decomposition), máy trạng thái "đã dùng lượt xóa hay chưa"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestAlternating(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Tính độ dài đoạn **luân phiên** dài nhất kết thúc tại mỗi vị trí, tách theo **chiều so sánh cuối** (`<` hay `>`).
2. Dùng kỹ thuật **tiền tố–hậu tố**: xóa phần tử `i` ⇔ nối đoạn kết thúc tại `i − 1` với đoạn bắt đầu tại `i + 1`, khi phép so sánh mới `nums[i−1]` vs `nums[i+1]` khớp chiều.
3. Viết lời giải **một lượt** bằng máy trạng thái có thêm cờ "đã xóa".
4. Xử lý cẩn thận phần tử **bằng nhau** (làm đứt đoạn luân phiên) và đoạn độ dài 1.
5. Kiểm thử với vét cạn "thử xóa từng phần tử".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Đoạn con `nums[l..r]` là **luân phiên** nếu các phép so sánh kề nhau xen kẽ giữa `>` và `<` (bắt đầu bằng chiều nào cũng được); đoạn độ dài 1 luôn luân phiên. Được **xóa tối đa một** phần tử khỏi mảng, sau đó chọn một đoạn luân phiên. Tìm độ dài lớn nhất.

**Ví dụ 1:** `[2,1,3,2]` → `4` (không xóa, `2 > 1 < 3 > 2`).

**Ví dụ 2:** `[3,2,1,2,3,2,1]` → `4` (xóa `nums[3] = 2`, mảng thành `[3,2,1,3,2,1]`; đoạn `[2,1,3,2]` luân phiên).

**Ví dụ 3:** `[100000,100000]` → `1`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`.

> Ghi chú: phần giải thích của đề ở ví dụ 2 viết "chọn `[3,2,1,3,2,1]`" nhưng dãy này **không** luân phiên (`3 > 2 > 1`); đoạn luân phiên dài nhất sau khi xóa là `[2,1,3,2]` hoặc `[1,3,2]`… với độ dài 4 — khớp đáp án `4`. Đáp án được kiểm chứng bằng vét cạn bên dưới.

In [1]:
from typing import List
import random, time

examples = [([2, 1, 3, 2], 4), ([3, 2, 1, 2, 3, 2, 1], 4), ([100000, 100000], 1)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[2, 1, 3, 2] -> kỳ vọng 4
[3, 2, 1, 2, 3, 2, 1] -> kỳ vọng 4
[100000, 100000] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Kiểm tra một dãy có luân phiên

Dùng dấu của hiệu: `sign(b − a)`. Dãy luân phiên ⇔ mọi hiệu khác 0 và hai hiệu liên tiếp trái dấu.

In [2]:
def is_alternating(a):
    d = [(b > x) - (b < x) for x, b in zip(a, a[1:])]
    return all(v != 0 for v in d) and all(d[i] != d[i + 1] for i in range(len(d) - 1))

for a in ([2, 1, 3, 2], [3, 2, 1, 3], [5], [4, 4], [1, 3, 2, 4]):
    print(a, is_alternating(a))

[2, 1, 3, 2] True
[3, 2, 1, 3] False
[5] True
[4, 4] False
[1, 3, 2, 4] True


### 3.2. Độ dài đoạn luân phiên dài nhất kết thúc tại `i`

`up[i]` = độ dài đoạn luân phiên dài nhất kết thúc tại `i` mà bước cuối là **tăng** (`nums[i−1] < nums[i]`); `down[i]` tương tự với bước cuối **giảm**. Một đoạn kết thúc bằng bước tăng phải nối tiếp đoạn kết thúc bằng bước giảm (hoặc một phần tử đơn):

- `nums[i−1] < nums[i]` ⇒ `up[i] = down[i−1] + 1`;
- `nums[i−1] > nums[i]` ⇒ `down[i] = up[i−1] + 1`;
- giá trị mặc định là `1` (đoạn một phần tử).

In [3]:
def ends_at(nums):
    n = len(nums); up = [1] * n; down = [1] * n
    for i in range(1, n):
        if nums[i - 1] < nums[i]: up[i] = down[i - 1] + 1
        elif nums[i - 1] > nums[i]: down[i] = up[i - 1] + 1
    return up, down

print(ends_at([3, 2, 1, 2, 3, 2, 1]))

([1, 1, 1, 3, 2, 1, 1], [1, 2, 2, 1, 1, 3, 2])


### 3.3. Đối xứng: đoạn **bắt đầu** tại `i`

Duyệt từ phải sang trái: `sup[i]` = độ dài đoạn luân phiên dài nhất **bắt đầu** tại `i` với bước đầu **tăng** (`nums[i] < nums[i+1]`), `sdown[i]` với bước đầu **giảm**:

- `nums[i] < nums[i+1]` ⇒ `sup[i] = sdown[i+1] + 1`;
- `nums[i] > nums[i+1]` ⇒ `sdown[i] = sup[i+1] + 1`.

In [4]:
def starts_at(nums):
    n = len(nums); sup = [1] * n; sdown = [1] * n
    for i in range(n - 2, -1, -1):
        if nums[i] < nums[i + 1]: sup[i] = sdown[i + 1] + 1
        elif nums[i] > nums[i + 1]: sdown[i] = sup[i + 1] + 1
    return sup, sdown

print(starts_at([3, 2, 1, 2, 3, 2, 1]))

([1, 1, 2, 3, 1, 1, 1], [2, 3, 1, 1, 2, 2, 1])


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `[3, 2, 1, 2, 3, 2, 1]`. Không xóa, dãy so sánh là `↓ ↓ ↑ ↑ ↓ ↓` — các đoạn luân phiên dài nhất chỉ dài 3 (ví dụ `[2, 1, 2]`, `[2, 3, 2]`).

Xóa `nums[3] = 2`: bên trái kết thúc tại chỉ số 2 (`[3, 2, 1]`… phần luân phiên cuối là `[2, 1]` với bước cuối **giảm**), bên phải bắt đầu tại chỉ số 4 (`[3, 2, 1]`, phần luân phiên đầu là `[3, 2]` với bước đầu **giảm**). Bước nối mới `1 → 3` là **tăng**: trước nó phải là giảm (✔ `2 > 1`), sau nó phải là giảm (✔ `3 > 2`). Ghép `[2, 1] + [3, 2]` = `[2, 1, 3, 2]` dài 4 ✔.

Quy tắc ghép khi xóa `i` (với `x = nums[i−1]`, `y = nums[i+1]`):

- `x < y` (bước nối tăng) ⇒ `down[i−1] + sdown[i+1]`;
- `x > y` (bước nối giảm) ⇒ `up[i−1] + sup[i+1]`;
- `x = y` ⇒ không nối được.

## 5. Cách 1 — Brute Force: thử xóa từng phần tử

Với mỗi lựa chọn (không xóa, hoặc xóa `i`), tính đoạn luân phiên dài nhất bằng một lượt quét `O(n)`. Tổng `O(n²)`.

In [5]:
def longest_alt(a):
    best = cur = 1
    for i in range(1, len(a)):
        if a[i] == a[i - 1]:
            cur = 1
        elif i >= 2 and a[i - 1] != a[i - 2] and (a[i] > a[i - 1]) != (a[i - 1] > a[i - 2]):
            cur += 1
        else:
            cur = 2
        best = max(best, cur)
    return best

def brute(nums):
    best = longest_alt(nums)
    for i in range(len(nums)):
        rest = nums[:i] + nums[i + 1:]
        if rest:
            best = max(best, longest_alt(rest))
    return best

for nums, exp in examples:
    print(brute(nums), "kỳ vọng", exp)

4 kỳ vọng 4
4 kỳ vọng 4
1 kỳ vọng 1


**Nút thắt:** `O(n²) = 10^{10}`. Nhưng việc xóa `i` chỉ ảnh hưởng **một chỗ nối** — phần còn lại đã tính sẵn ở tiền tố/hậu tố.

## 6. Ý tưởng thuật toán / bước đột phá

**Không xóa:** đáp án ứng viên `max(up[i], down[i])`.

**Xóa `i` (`1 ≤ i ≤ n−2`):** đoạn tốt nhất **đi qua chỗ xóa** gồm phần cuối của đoạn kết thúc tại `i−1` và phần đầu của đoạn bắt đầu tại `i+1`, nối bằng bước `nums[i−1] → nums[i+1]`. Để toàn bộ luân phiên, bước trước chỗ nối phải ngược chiều bước nối, và bước sau chỗ nối cũng ngược chiều. Vì vậy dùng `down/up` ở bên trái và `sdown/sup` ở bên phải như quy tắc mục 4.

Xóa phần tử ở **hai đầu** không bao giờ tạo đoạn dài hơn trường hợp không xóa (chỉ làm ngắn mảng) ⇒ bỏ qua.

Đáp án = max của mọi ứng viên. Tất cả `O(n)`.

## 7. Các cách tiếp theo

- **Cách 2:** máy trạng thái **một lượt**: `A[i][dir]` (chưa xóa) và `B[i][dir]` (đã xóa một phần tử, phần tử cuối là `i`) — `O(n)`.
- **Cách 3:** tiền tố `up/down` + hậu tố `sup/sdown` + thử mọi chỗ xóa (lời giải nộp chính) — `O(n)`.

### 7.1. Cách 2 — Máy trạng thái một lượt

**Trạng thái** tại `i` (đoạn kết thúc tại `i`):

- `A_up, A_down`: chưa xóa; bước cuối tăng/giảm (giống `up/down`).
- `B_up, B_down`: đã xóa đúng một phần tử **bên trong** đoạn; bước cuối tăng/giảm.

**Chuyển trạng thái** tới `i`:

- từ `i − 1` (không xóa gì mới): `B_up[i] = B_down[i−1] + 1` nếu `nums[i−1] < nums[i]` (tương tự cho giảm); `A` như cũ.
- **xóa `i − 1`** (nối `i − 2` với `i`): nếu `nums[i−2] < nums[i]` thì `B_up[i] = A_down[i−2] + 1`; nếu `>` thì `B_down[i] = A_up[i−2] + 1`.

Đáp án = max mọi trạng thái.

In [6]:
def state_machine(nums):
    n = len(nums)
    Au, Ad = [1] * n, [1] * n
    Bu, Bd = [0] * n, [0] * n           # 0 = chưa hợp lệ
    best = 1
    for i in range(1, n):
        x, y = nums[i - 1], nums[i]
        if x < y:
            Au[i] = Ad[i - 1] + 1
            if Bd[i - 1]: Bu[i] = Bd[i - 1] + 1
        elif x > y:
            Ad[i] = Au[i - 1] + 1
            if Bu[i - 1]: Bd[i] = Bu[i - 1] + 1
        if i >= 2:                       # xóa nums[i-1]
            z = nums[i - 2]
            if z < y: Bu[i] = max(Bu[i], Ad[i - 2] + 1)
            elif z > y: Bd[i] = max(Bd[i], Au[i - 2] + 1)
        best = max(best, Au[i], Ad[i], Bu[i], Bd[i])
    return best

for nums, exp in examples:
    print(state_machine(nums), "kỳ vọng", exp)

4 kỳ vọng 4
4 kỳ vọng 4
1 kỳ vọng 1


In [7]:
class SolutionStateMachine:
    def longestAlternating(self, nums: List[int]) -> int:
        return state_machine(nums)

print([SolutionStateMachine().longestAlternating(a) for a, _ in examples])

[4, 4, 1]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ (có thể giảm `O(1)` vì chỉ cần hai vị trí trước).

**Lưu ý:** `B` chỉ dùng khi phần tử bị xóa nằm **giữa** đoạn; xóa ngoài đoạn tương đương "không xóa" và đã được `A` tính.

### 7.2. Cách 3 — Tiền tố + hậu tố (lời giải chính)

**Giải thích từng đoạn:**

1. `up, down` từ trái sang (mục 3.2); `ans = max(up, down)`.
2. `sup, sdown` từ phải sang (mục 3.3).
3. Với `i = 1..n−2`: nếu `nums[i−1] < nums[i+1]` ⇒ ứng viên `down[i−1] + sdown[i+1]`; nếu `>` ⇒ `up[i−1] + sup[i+1]`.

In [8]:
def prefix_suffix(nums):
    n = len(nums)
    up, down = ends_at(nums)
    sup, sdown = starts_at(nums)
    ans = max(max(up), max(down))
    for i in range(1, n - 1):
        x, y = nums[i - 1], nums[i + 1]
        if x < y:
            ans = max(ans, down[i - 1] + sdown[i + 1])
        elif x > y:
            ans = max(ans, up[i - 1] + sup[i + 1])
    return ans

for nums, exp in examples:
    print(prefix_suffix(nums), "kỳ vọng", exp)

4 kỳ vọng 4
4 kỳ vọng 4
1 kỳ vọng 1


**Độ phức tạp Cách 3:** ba lượt `O(n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 `[3, 2, 1, 2, 3, 2, 1]`.

In [9]:
def trace(nums):
    n = len(nums)
    up, down = ends_at(nums); sup, sdown = starts_at(nums)
    print("i     :", list(range(n)))
    print("nums  :", nums)
    print("up    :", up); print("down  :", down)
    print("sup   :", sup); print("sdown :", sdown)
    ans = max(max(up), max(down)); print("không xóa:", ans)
    for i in range(1, n - 1):
        x, y = nums[i - 1], nums[i + 1]
        if x < y:
            c = down[i - 1] + sdown[i + 1]; how = f"nối tăng {x}<{y}: down[{i-1}]+sdown[{i+1}]"
        elif x > y:
            c = up[i - 1] + sup[i + 1]; how = f"nối giảm {x}>{y}: up[{i-1}]+sup[{i+1}]"
        else:
            c = 0; how = f"{x}={y}: không nối được"
        ans = max(ans, c)
        print(f"xóa i={i}: {how} = {c}")
    print("đáp án:", ans)

trace([3, 2, 1, 2, 3, 2, 1])

i     : [0, 1, 2, 3, 4, 5, 6]
nums  : [3, 2, 1, 2, 3, 2, 1]
up    : [1, 1, 1, 3, 2, 1, 1]
down  : [1, 2, 2, 1, 1, 3, 2]
sup   : [1, 1, 2, 3, 1, 1, 1]
sdown : [2, 3, 1, 1, 2, 2, 1]
không xóa: 3
xóa i=1: nối giảm 3>1: up[0]+sup[2] = 3
xóa i=2: 2=2: không nối được = 0
xóa i=3: nối tăng 1<3: down[2]+sdown[4] = 4
xóa i=4: 2=2: không nối được = 0
xóa i=5: nối giảm 3>1: up[4]+sup[6] = 3
đáp án: 4


Xóa `i = 3` cho `down[2] + sdown[4] = 2 + 2 = 4` (đoạn `[2, 1] + [3, 2]`). Các chỗ xóa khác chỉ cho 3 (ví dụ xóa `i = 1`: `up[0] + sup[2] = 1 + 2 = 3`, đoạn `[3] + [1, 2]`), còn `i = 2, 4` có hai phần tử kề mới bằng nhau nên không nối được. Ô dưới kiểm chứng bằng cách xóa thật.

In [10]:
# Kiểm chứng từng chỗ xóa bằng vét cạn
nums = [3, 2, 1, 2, 3, 2, 1]
for i in range(len(nums)):
    rest = nums[:i] + nums[i + 1:]
    print(f"xóa {i}: {rest} -> dài nhất {longest_alt(rest)}")

xóa 0: [2, 1, 2, 3, 2, 1] -> dài nhất 3
xóa 1: [3, 1, 2, 3, 2, 1] -> dài nhất 3
xóa 2: [3, 2, 2, 3, 2, 1] -> dài nhất 3
xóa 3: [3, 2, 1, 3, 2, 1] -> dài nhất 4
xóa 4: [3, 2, 1, 2, 2, 1] -> dài nhất 3
xóa 5: [3, 2, 1, 2, 3, 1] -> dài nhất 3
xóa 6: [3, 2, 1, 2, 3, 2] -> dài nhất 3


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def longestAlternating(self, nums: List[int]) -> int:
        n = len(nums)
        up = [1] * n; down = [1] * n        # kết thúc tại i, bước cuối tăng / giảm
        for i in range(1, n):
            if nums[i - 1] < nums[i]:
                up[i] = down[i - 1] + 1
            elif nums[i - 1] > nums[i]:
                down[i] = up[i - 1] + 1
        sup = [1] * n; sdown = [1] * n      # bắt đầu tại i, bước đầu tăng / giảm
        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1]:
                sup[i] = sdown[i + 1] + 1
            elif nums[i] > nums[i + 1]:
                sdown[i] = sup[i + 1] + 1
        ans = max(max(up), max(down))
        for i in range(1, n - 1):
            x, y = nums[i - 1], nums[i + 1]
            if x < y:
                ans = max(ans, down[i - 1] + sdown[i + 1])
            elif x > y:
                ans = max(ans, up[i - 1] + sup[i + 1])
        return ans

sol = Solution()
for nums, exp in examples:
    got = sol.longestAlternating(nums)
    print(nums, "->", got, "✔" if got == exp else "✘")

[2, 1, 3, 2] -> 4 ✔
[3, 2, 1, 2, 3, 2, 1] -> 4 ✔
[100000, 100000] -> 1 ✔


## 10. Vì sao thuật toán đúng

1. **Không xóa:** mọi đoạn luân phiên kết thúc tại `i` có bước cuối tăng hoặc giảm (hoặc dài 1); `up/down` tính đúng độ dài lớn nhất nhờ quy nạp: bỏ phần tử cuối của đoạn kết thúc bằng bước tăng được đoạn kết thúc tại `i−1` bằng bước giảm (hoặc đơn).
2. **Có xóa `i`:** đoạn được chọn sau khi xóa hoặc **không chứa** chỗ nối `i−1 → i+1` (khi đó nó là đoạn của mảng gốc — đã tính), hoặc chứa nó. Nếu chứa, nó gồm phần bên trái (kết thúc tại `i−1`) và phần bên phải (bắt đầu tại `i+1`), mỗi phần tự luân phiên; điều kiện khớp chiều tại chỗ nối đúng như quy tắc. Phần trái dài nhất với chiều cuối cho trước chính là `down[i−1]`/`up[i−1]` (độc lập với phần phải), tương tự phần phải.
3. **Đoạn dài 1 ở một phía** (ví dụ `down[i−1] = 1`) vẫn đúng: phần đó chỉ là `nums[i−1]`, không có bước nào để kiểm tra.
4. **Xóa ở đầu/cuối mảng** chỉ loại bỏ phần tử, không tạo bước so sánh mới ⇒ không cải thiện.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử xóa từng phần tử | `O(n²)` | `O(n)` |
| 2. Máy trạng thái | `O(n)` | `O(n)` (có thể `O(1)`) |
| 3. Tiền tố + hậu tố | `O(n)` | `O(n)` |

## 12. Bộ test

In [12]:
def run_all(nums, with_brute=True):
    res = [state_machine(nums), prefix_suffix(nums), Solution().longestAlternating(nums)]
    if with_brute:
        res.append(brute(nums))
    return res

for nums, exp in examples:
    assert all(v == exp for v in run_all(nums))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([1, 2], 2), ([5, 5, 5], 1),
    ([1, 2, 3], 2),                      # tăng dần: xóa giữa vẫn chỉ được 2
    ([1, 3, 3, 2], 3),                   # xóa một số 3 -> [1,3,2]
    ([1, 3, 2, 2, 1, 3], 3),             # xóa một số 2 -> [1,3,2,1,3] vẫn có 3>2>1: không lên được 4
    ([1, 5, 2, 6, 3, 7], 6),             # đã luân phiên
]
for nums, exp in cases:
    r = run_all(nums)
    print(nums, "->", r[0], "| vét cạn:", r[-1])
    assert len(set(r)) == 1 and r[0] == exp

[1, 2] -> 2 | vét cạn: 2
[5, 5, 5] -> 1 | vét cạn: 1
[1, 2, 3] -> 2 | vét cạn: 2
[1, 3, 3, 2] -> 3 | vét cạn: 3
[1, 3, 2, 2, 1, 3] -> 3 | vét cạn: 3
[1, 5, 2, 6, 3, 7] -> 6 | vét cạn: 6


In [14]:
# 12.3 phản ví dụ: chỉ cộng độ dài hai bên mà không kiểm tra chiều tại chỗ nối
def wrong_no_direction(nums):
    n = len(nums); up, down = ends_at(nums); sup, sdown = starts_at(nums)
    ans = max(max(up), max(down))
    for i in range(1, n - 1):
        if nums[i - 1] != nums[i + 1]:
            ans = max(ans, max(up[i - 1], down[i - 1]) + max(sup[i + 1], sdown[i + 1]))   # SAI
    return ans

rng = random.Random(1)
for _ in range(5000):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(3, 8))]
    if wrong_no_direction(nums) != Solution().longestAlternating(nums):
        print("phản ví dụ:", nums, "| sai:", wrong_no_direction(nums), "| đúng:", Solution().longestAlternating(nums)); break

phản ví dụ: [4, 4, 2, 1, 4, 1] | sai: 5 | đúng: 4


In [15]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3830)
for _ in range(800):
    nums = [rng.randint(1, rng.choice([2, 3, 5, 10])) for _ in range(rng.randint(2, 14))]
    r = run_all(nums)
    assert len(set(r)) == 1, (nums, r)
print("800 test ngẫu nhiên: OK")

800 test ngẫu nhiên: OK


In [16]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
nums = [rng.randint(1, 10**5) for _ in range(10**5)]
t0 = time.time(); a = Solution().longestAlternating(nums); t1 = time.time()
print("ngẫu nhiên:", a, "| khớp máy trạng thái:", a == state_machine(nums), "| thời gian < 1s:", t1 - t0 < 1)
zig = [1 + (i % 2) for i in range(10**5)]
zig[50000] = zig[49999]                     # tạo một chỗ "hỏng" ở giữa
print("zigzag có ba số 2 liền nhau ở giữa (xóa một vẫn còn hai số bằng nhau):", Solution().longestAlternating(zig), "| vét cạn trên bản thu nhỏ:", brute(zig[49990:50010]) == Solution().longestAlternating(zig[49990:50010]))

ngẫu nhiên: 30 | khớp máy trạng thái: True | thời gian < 1s: True


zigzag có ba số 2 liền nhau ở giữa (xóa một vẫn còn hai số bằng nhau): 50000 | vét cạn trên bản thu nhỏ: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không kiểm tra chiều tại chỗ nối

Đã thấy ở 12.3: ghép hai phía mà không quan tâm bước trước/sau chỗ nối có ngược chiều không sẽ đếm cả dãy kiểu `↓ ↑ ↑`.

### Lỗi 2 — Coi hai phần tử bằng nhau là "luân phiên"

Hai phần tử bằng nhau **làm đứt** đoạn. Nếu viết `if nums[i-1] <= nums[i]` (gộp bằng vào tăng) thì đếm sai.

In [17]:
def wrong_equal(nums):
    n = len(nums); up = [1] * n; down = [1] * n
    for i in range(1, n):
        if nums[i - 1] <= nums[i]: up[i] = down[i - 1] + 1      # SAI: <=
        else: down[i] = up[i - 1] + 1
    return max(max(up), max(down))

print("sai:", wrong_equal([100000, 100000]), "| đúng:", Solution().longestAlternating([100000, 100000]))

sai: 2 | đúng: 1


### Lỗi 3 — Nhầm `sup` / `sdown` ở phía phải

Bước nối **tăng** (`x < y`) thì bước tiếp theo phải **giảm** ⇒ phía phải phải **bắt đầu bằng bước giảm** (`sdown`), không phải `sup`.

In [18]:
def wrong_swap(nums):
    n = len(nums); up, down = ends_at(nums); sup, sdown = starts_at(nums)
    ans = max(max(up), max(down))
    for i in range(1, n - 1):
        x, y = nums[i - 1], nums[i + 1]
        if x < y: ans = max(ans, down[i - 1] + sup[i + 1])       # SAI: phải là sdown
        elif x > y: ans = max(ans, up[i - 1] + sdown[i + 1])     # SAI: phải là sup
    return ans

print("sai:", wrong_swap([1, 2, 1, 3, 4, 5]), "| đúng:", Solution().longestAlternating([1, 2, 1, 3, 4, 5]), "| vét cạn:", brute([1, 2, 1, 3, 4, 5]))

sai: 5 | đúng: 4 | vét cạn: 4


### Lỗi 4 — Khởi tạo `up/down` bằng 0

Đoạn một phần tử có độ dài 1; nếu khởi tạo 0, đoạn hai phần tử chỉ được tính là 1 và mảng toàn bằng nhau trả `0`.

In [19]:
def wrong_zero(nums):
    n = len(nums); up = [0] * n; down = [0] * n
    for i in range(1, n):
        if nums[i - 1] < nums[i]: up[i] = down[i - 1] + 1
        elif nums[i - 1] > nums[i]: down[i] = up[i - 1] + 1
    return max(max(up), max(down))

print("sai:", wrong_zero([5, 5, 5]), wrong_zero([1, 2]), "| đúng:", Solution().longestAlternating([5, 5, 5]), Solution().longestAlternating([1, 2]))

sai: 0 1 | đúng: 1 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Được xóa tối đa một phần tử"** ⇒ tiền tố (kết thúc tại `i−1`) + hậu tố (bắt đầu tại `i+1`) + điều kiện tại chỗ nối; hoặc thêm cờ "đã xóa" vào trạng thái DP (LC 1186 *Maximum Subarray Sum with One Deletion*, LC 1493 *Longest Subarray of 1's After Deleting One Element*).
- **Luân phiên** ⇒ tách trạng thái theo chiều của bước cuối.
- Phần tử bằng nhau là "điểm gãy" — xử lý riêng.

## 15. Biến thể của bài

1. **Không được xóa**: LC 978 *Longest Turbulent Subarray* — chỉ `up/down`.
2. **Xóa tối đa `k` phần tử** (`k` nhỏ): trạng thái `(i, dir, số lần xóa)` với chuyển "bỏ qua tối đa vài phần tử liền nhau".
3. **Dãy con** (không cần liên tiếp) luân phiên dài nhất: LC 376 *Wiggle Subsequence* — tham lam đếm số lần đổi chiều.
4. **Được thay đổi** một phần tử thành giá trị bất kỳ (thay vì xóa).

In [20]:
# Biến thể 3: Wiggle Subsequence (dãy con, không cần liên tiếp)
def wiggle_subsequence(nums):
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]: up = down + 1
        elif nums[i] < nums[i - 1]: down = up + 1
    return max(up, down)

print(wiggle_subsequence([1, 7, 4, 9, 2, 5]), wiggle_subsequence([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]), "(LC 376: 6, 7)")

6 7 (LC 376: 6, 7)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử xóa từng phần tử | Quét lại toàn bộ | `O(n²)` | `O(n)` | Kiểm thử |
| 2. Máy trạng thái | Cờ "đã xóa" trong DP | `O(n)` | `O(1)`–`O(n)` | Mở rộng sang "xóa `k`" |
| 3. Tiền tố + hậu tố | Ghép tại chỗ nối | `O(n)` | `O(n)` | **Lời giải chuẩn**, dễ suy luận |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `longestAlternating([1, 1, 2, 1])` = ?

**Bài 2 (tính tay).** `[4, 2, 2, 5, 1]` — lập bảng `up/down/sup/sdown` và thử mọi chỗ xóa.

**Bài 3 (điền TODO).** Viết `longest_turbulent(nums)` (không xóa).

**Bài 4 (tự viết lại).** Viết lại Cách 2 với bộ nhớ `O(1)` (chỉ giữ giá trị của hai vị trí trước).

**Bài 5 (mini challenge).** Cho phép xóa tối đa **hai** phần tử (bất kỳ). Viết DP và kiểm chứng bằng vét cạn.

In [21]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def longest_turbulent(nums):
    # TODO: up/down như ends_at, trả về max
    pass

In [22]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def at_most_two_deletions(nums):
    # TODO: trạng thái (i, chiều cuối, số lần xóa 0..2); chuyển từ i-1, i-2 (xóa 1), i-3 (xóa 2 liền nhau)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không xóa: `[1, 2, 1]` dài 3. (2) Xóa một số 1 đầu không giúp thêm. (3) Đáp án `3`.

**Bài 2.** (1) Chỗ hỏng là `2, 2`. (2) Xóa một số 2: `[4, 2, 5, 1]` luân phiên dài 4. (3) Đáp án `4`.

**Bài 3.** (1) Như `ends_at`. (2) Lấy max. (3) LC 978 ví dụ `[9,4,2,10,7,8,8,1,9]` → `5`.

**Bài 4.** (1) Biến cho `A_up, A_down` tại `i−1`, `i−2`. (2) Cập nhật cuốn chiếu. (3) Đối chiếu.

**Bài 5.** (1) Hai lần xóa có thể liền nhau (nối `i−3` với `i`) hoặc tách rời. (2) Trạng thái số lần xóa 0, 1, 2. (3) Vét cạn mọi cặp chỉ số bị xóa.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().longestAlternating([1, 1, 2, 1]) == 3 == brute([1, 1, 2, 1])
assert Solution().longestAlternating([4, 2, 2, 5, 1]) == 4 == brute([4, 2, 2, 5, 1])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [24]:
# Bài 3
def longest_turbulent_sol(nums):
    up, down = ends_at(nums)
    return max(max(up), max(down))

assert longest_turbulent_sol([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5
assert longest_turbulent_sol([4, 8, 12, 16]) == 2
print("Bài 3 OK")

Bài 3 OK


In [25]:
# Bài 4: O(1) bộ nhớ
def state_machine_o1(nums):
    n = len(nums)
    au1 = ad1 = 1; bu1 = bd1 = 0          # tại i-1
    au2 = ad2 = 1                          # A tại i-2
    best = 1
    for i in range(1, n):
        x, y = nums[i - 1], nums[i]
        au = ad = 1; bu = bd = 0
        if x < y:
            au = ad1 + 1
            if bd1: bu = bd1 + 1
        elif x > y:
            ad = au1 + 1
            if bu1: bd = bu1 + 1
        if i >= 2:
            z = nums[i - 2]
            if z < y: bu = max(bu, ad2 + 1)
            elif z > y: bd = max(bd, au2 + 1)
        best = max(best, au, ad, bu, bd)
        au2, ad2 = au1, ad1
        au1, ad1, bu1, bd1 = au, ad, bu, bd
    return best

rng = random.Random(4)
for _ in range(500):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(2, 12))]
    assert state_machine_o1(nums) == Solution().longestAlternating(nums)
print("Bài 4 OK")

Bài 4 OK


In [26]:
# Bài 5: xóa tối đa hai phần tử — DP trạng thái f[i][dir][d] (đoạn kết thúc tại i, đã xóa d phần tử bên trong)
def at_most_two_sol(nums):
    n = len(nums)
    f = [[[0] * 3 for _ in range(2)] for _ in range(n)]     # dir 0 = tăng, 1 = giảm; 0 = chưa hợp lệ
    best = 1
    for i in range(n):
        for d in range(3):
            for skip in range(d + 1):                          # bỏ skip phần tử ngay trước i
                j = i - 1 - skip
                if j < 0: continue
                if nums[j] < nums[i]:
                    prev = max(f[j][1][d - skip], 1)
                    f[i][0][d] = max(f[i][0][d], prev + 1)
                elif nums[j] > nums[i]:
                    prev = max(f[j][0][d - skip], 1)
                    f[i][1][d] = max(f[i][1][d], prev + 1)
            best = max(best, f[i][0][d], f[i][1][d])
    return best

def brute_two(nums):
    best = longest_alt(nums); n = len(nums)
    for i in range(n):
        for j in range(i, n):
            rest = [x for t, x in enumerate(nums) if t != i and t != j]
            if rest: best = max(best, longest_alt(rest))
    return best

rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(2, 10))]
    assert at_most_two_sol(nums) == brute_two(nums), nums
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
up[i]   = down[i-1] + 1 if nums[i-1] < nums[i] else 1
down[i] = up[i-1]   + 1 if nums[i-1] > nums[i] else 1
sup[i]  = sdown[i+1] + 1 if nums[i] < nums[i+1] else 1
sdown[i]= sup[i+1]   + 1 if nums[i] > nums[i+1] else 1
ans = max(up + down)
for i in 1..n-2:
    if nums[i-1] < nums[i+1]: ans = max(ans, down[i-1] + sdown[i+1])
    if nums[i-1] > nums[i+1]: ans = max(ans, up[i-1] + sup[i+1])
```

**Checklist:**

- [ ] Giá trị khởi tạo `1` (đoạn một phần tử)?
- [ ] Phần tử bằng nhau làm đứt đoạn?
- [ ] Nối tăng ⇒ trái kết thúc **giảm**, phải bắt đầu **giảm**; nối giảm ⇒ ngược lại?
- [ ] Bỏ qua chỗ nối có `nums[i−1] == nums[i+1]`?
- [ ] Gồm cả phương án **không xóa**?